# Workshop 6 — Medical Domain Adaptation
## 02. Transfer Learning: Classical Baselines + MLP + Autoencoder

**Course:** Machine Learning — Semester 2026-III
**Team:** Group 1 — Housing Price Prediction → Cardiac Arrhythmia Detection

This notebook implements the FASE 2 transfer pipeline: adapting the
techniques from Workshops 1, 2, and 4 to ECG arrhythmia classification.

### Transfer strategy

| Workshop | Original domain | Medical adaptation | Transferred component |
|----------|-----------------|--------------------|-----------------------|
| W1 | Housing tabular regression | Beat-level binary classification | Classical baselines (LogReg, RF, GBR) |
| W2 | 261→128→64→1 MLP regression | 7→128→64→2 MLP classification | Architecture + conditioning protocol + class weighting |
| W4 | AE on housing features | AE trained on Normal beats only | Anomaly scoring by reconstruction error |

### Evaluation protocol

- **Split:** patient-level (34 train / 7 val / 7 test records)
- **Preprocessing:** two-stage scaler (Robust + Standard), fit on TRAIN ONLY
- **Class imbalance:** `class_weight='balanced'` (W1) and inverse-frequency loss (W2)
- **Multi-seed:** {7, 42, 2026} for stability estimation
- **Threshold:** tuned on VALIDATION, then applied once to TEST
- **Clinical metrics:** sensitivity, specificity, PPV, NPV at clinically-relevant thresholds

In [6]:
import sys
sys.path.append("../src")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from medical_preprocessing import FEATURE_COLUMNS, load_splits, fit_scalers, transform
from transfer import (
    set_seed, MLPConfig,
    train_logistic_regression, train_random_forest, train_gradient_boosting,
    train_mlp, predict_mlp_proba, MedicalMLP,
    evaluate_binary, threshold_sweep,
)

set_seed(42)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device: cpu
PyTorch: 2.14.1+cpu


## 1. Data Loading & Leakage-Aware Scaling

Load the preprocessed feature matrix and the patient-level stratified split,
then fit the two-stage scaler (Robust + Standard) **exclusively on the training
partition**. The fitted scalers are then applied unchanged to validation and test.

This mirrors the Workshop 2 protocol, which showed that target and input
standardization were critical for stable MLP training.

In [7]:
# Load feature matrix and splits
df = pd.read_csv("../data/features_mitdb.csv")
df["record_id"] = df["record_id"].astype(str)

splits = load_splits("../data/splits_medical.json")

# Build train/val/test arrays
def subset(records):
    sub = df[df["record_id"].isin(records)].reset_index(drop=True)
    X = sub[FEATURE_COLUMNS].values.astype(np.float32)
    y = sub["label_binary"].values.astype(np.int64)
    return X, y, sub

X_train, y_train, df_train = subset(splits.train_records)
X_val,   y_val,   df_val   = subset(splits.val_records)
X_test,  y_test,  df_test  = subset(splits.test_records)

print(f"Train: {X_train.shape} | N={int((y_train==0).sum()):,} | Ab={int((y_train==1).sum()):,}")
print(f"Val:   {X_val.shape}   | N={int((y_val==0).sum()):,} | Ab={int((y_val==1).sum()):,}")
print(f"Test:  {X_test.shape}  | N={int((y_test==0).sum()):,} | Ab={int((y_test==1).sum()):,}")

# Fit scalers on TRAIN ONLY
scalers = fit_scalers(X_train)
X_train_s = transform(X_train, scalers)
X_val_s   = transform(X_val, scalers)
X_test_s  = transform(X_test, scalers)

print(f"\nAfter scaling — train mean: {X_train_s.mean(axis=0).round(3)}")
print(f"After scaling — train std:  {X_train_s.std(axis=0).round(3)}")

Train: (69455, 8) | N=61,972 | Ab=7,483
Val:   (16541, 8)   | N=14,416 | Ab=2,125
Test:  (15301, 8)  | N=14,100 | Ab=1,201

After scaling — train mean: [ 0. -0.  0. -0. -0.  0. -0. -0.]
After scaling — train std:  [1. 1. 1. 1. 1. 1. 1. 1.]


## 2. Workshop 1 Transfer: Classical Baselines

We re-train the three Workshop 1 baselines — Logistic Regression, Random Forest,
and Gradient Boosting — on the medical feature matrix. All use `class_weight='balanced'`
to counter the 8.4:1 imbalance. This establishes the reference against which the
Workshop 2 MLP will be compared.

In [8]:
import time

baselines = {}
baseline_results = []

# Logistic Regression
t0 = time.time()
lr_model = train_logistic_regression(X_train_s, y_train)
lr_proba_val = lr_model.predict_proba(X_val_s)[:, 1]
lr_metrics = evaluate_binary(y_val, lr_proba_val, threshold=0.5)
lr_metrics["model"] = "LogisticRegression"
lr_metrics["train_time_s"] = time.time() - t0
baselines["LogReg"] = (lr_model, lr_proba_val)
baseline_results.append(lr_metrics)
print(f"LogReg  | val AUC={lr_metrics['roc_auc']:.4f} | PR-AUC={lr_metrics['pr_auc']:.4f} | {lr_metrics['train_time_s']:.2f}s")

# Random Forest
t0 = time.time()
rf_model = train_random_forest(X_train_s, y_train)
rf_proba_val = rf_model.predict_proba(X_val_s)[:, 1]
rf_metrics = evaluate_binary(y_val, rf_proba_val, threshold=0.5)
rf_metrics["model"] = "RandomForest"
rf_metrics["train_time_s"] = time.time() - t0
baselines["RandomForest"] = (rf_model, rf_proba_val)
baseline_results.append(rf_metrics)
print(f"RF      | val AUC={rf_metrics['roc_auc']:.4f} | PR-AUC={rf_metrics['pr_auc']:.4f} | {rf_metrics['train_time_s']:.2f}s")

# Gradient Boosting
t0 = time.time()
gb_model = train_gradient_boosting(X_train_s, y_train)
gb_proba_val = gb_model.predict_proba(X_val_s)[:, 1]
gb_metrics = evaluate_binary(y_val, gb_proba_val, threshold=0.5)
gb_metrics["model"] = "GradientBoosting"
gb_metrics["train_time_s"] = time.time() - t0
baselines["GradientBoosting"] = (gb_model, gb_proba_val)
baseline_results.append(gb_metrics)
print(f"GBR     | val AUC={gb_metrics['roc_auc']:.4f} | PR-AUC={gb_metrics['pr_auc']:.4f} | {gb_metrics['train_time_s']:.2f}s")

LogReg  | val AUC=0.8378 | PR-AUC=0.5249 | 0.08s
RF      | val AUC=0.9144 | PR-AUC=0.7465 | 1.78s
GBR     | val AUC=0.9210 | PR-AUC=0.6063 | 23.17s


### 2.1 Baseline Summary (Validation)

Metrics at default threshold (0.5). Note that PR-AUC is more informative than
ROC-AUC here because of the class imbalance: a high ROC-AUC can still hide poor
performance on the positive class.

In [9]:
baseline_df = pd.DataFrame(baseline_results)[
    ["model", "roc_auc", "pr_auc", "sensitivity", "specificity",
     "ppv", "npv", "f1", "train_time_s"]
].round(4)
print(baseline_df.to_string(index=False))

             model  roc_auc  pr_auc  sensitivity  specificity    ppv    npv     f1  train_time_s
LogisticRegression   0.8378  0.5249       0.7078       0.8601 0.4272 0.9523 0.5328        0.0798
      RandomForest   0.9144  0.7465       0.7454       0.9052 0.5369 0.9602 0.6242        1.7806
  GradientBoosting   0.9210  0.6063       0.6899       0.9221 0.5662 0.9528 0.6220       23.1717


## 3. Workshop 2 Transfer: MLP with Class Weighting

The Workshop 2 MLP (261→128→64→1 regression) is re-purposed here as a
binary classifier (7→128→64→2 classification). We transfer:
- **Architecture:** 128-64 hidden dims, dropout 0.05 (same as W2).
- **Conditioning:** two-stage scaler (Robust + Standard), already applied.
- **Class imbalance handling:** inverse-frequency loss weighting.
- **Training protocol:** AdamW (lr=1e-3, wd=1e-5), grad clip=5, early stopping on val AUC with patience 15.

We train with three seeds (7, 42, 2026) to estimate stability.

In [10]:
from transfer import MLPConfig

config = MLPConfig(
    input_dim=X_train_s.shape[1],
    hidden_dims=(128, 64),
    dropout=0.05,
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=256,
    max_epochs=100,
    patience=15,
)

seeds = [7, 42, 2026]
mlp_runs = []  # store (seed, model, history, val_proba)

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"Training MLP with seed={seed}")
    print(f"{'='*60}")
    set_seed(seed)
    model, history = train_mlp(
        X_train_s, y_train,
        X_val_s, y_val,
        config=config, seed=seed, verbose=True,
    )
    val_proba = predict_mlp_proba(model, X_val_s)
    metrics = evaluate_binary(y_val, val_proba, threshold=0.5)
    metrics["seed"] = seed
    metrics["best_val_auc"] = max(history["val_auc"])
    metrics["epochs_run"] = len(history["val_auc"])
    mlp_runs.append({"seed": seed, "model": model, "history": history,
                     "val_proba": val_proba, "metrics": metrics})
    print(f"Seed {seed}: best val AUC={metrics['best_val_auc']:.4f}, "
          f"PR-AUC={metrics['pr_auc']:.4f}, epochs={metrics['epochs_run']}")


Training MLP with seed=7
Epoch   1 | train_loss=0.3350 | val_loss=0.4338 | val_AUC=0.9157
Epoch   5 | train_loss=0.1376 | val_loss=0.4728 | val_AUC=0.9246
Epoch  10 | train_loss=0.1188 | val_loss=0.4929 | val_AUC=0.9259
Epoch  15 | train_loss=0.1103 | val_loss=0.5225 | val_AUC=0.9232
Epoch  20 | train_loss=0.1072 | val_loss=0.5389 | val_AUC=0.9195
Epoch  25 | train_loss=0.1030 | val_loss=0.5272 | val_AUC=0.9210
Early stopping at epoch 25 (best AUC=0.9259)
Seed 7: best val AUC=0.9259, PR-AUC=0.5977, epochs=25

Training MLP with seed=42
Epoch   1 | train_loss=0.3258 | val_loss=0.4437 | val_AUC=0.9148
Epoch   5 | train_loss=0.1347 | val_loss=0.4841 | val_AUC=0.9207
Epoch  10 | train_loss=0.1149 | val_loss=0.5615 | val_AUC=0.9137
Epoch  15 | train_loss=0.1043 | val_loss=0.5231 | val_AUC=0.9219
Epoch  20 | train_loss=0.0985 | val_loss=0.5359 | val_AUC=0.9182
Early stopping at epoch 21 (best AUC=0.9238)
Seed 42: best val AUC=0.9238, PR-AUC=0.5915, epochs=21

Training MLP with seed=2026
Epoc

## 4. Threshold Analysis: Choosing the Clinical Operating Point

In medical AI, the default threshold (0.5) is arbitrary and rarely optimal.
Clinicians care about **sensitivity** (did we catch the disease?) and
**specificity** (did we avoid false alarms?). The right operating point
depends on the clinical cost of a missed arrhythmia vs an alarm.

We sweep thresholds from 0.05 to 0.95 on the VALIDATION set (never on test)
and select the point that maximizes F1 or achieves a clinically reasonable
sensitivity (e.g., ≥ 80%). The chosen threshold is then applied once to test.

In [11]:
# Use the best seed for threshold analysis
best_run = max(mlp_runs, key=lambda r: r["metrics"]["roc_auc"])
print(f"Using seed {best_run['seed']} (val AUC={best_run['metrics']['roc_auc']:.4f})")

# Also sweep Random Forest for comparison
rf_sweep = threshold_sweep(y_val, rf_proba_val)
mlp_sweep = threshold_sweep(y_val, best_run["val_proba"])

# Add model label
rf_sweep["model"] = "RandomForest"
mlp_sweep["model"] = "MLP"

# Print selected thresholds
for name, sweep in [("RandomForest", rf_sweep), ("MLP", mlp_sweep)]:
    print(f"\n=== {name} — top 5 thresholds by F1 ===")
    print(sweep.nlargest(5, "f1")[["threshold", "sensitivity", "specificity", "ppv", "npv", "f1"]].round(4).to_string(index=False))

Using seed 2026 (val AUC=0.9268)

=== RandomForest — top 5 thresholds by F1 ===
 threshold  sensitivity  specificity    ppv    npv     f1
      0.85       0.6348       0.9618 0.7100 0.9470 0.6703
      0.90       0.5722       0.9763 0.7810 0.9393 0.6605
      0.80       0.6555       0.9505 0.6614 0.9493 0.6585
      0.75       0.6729       0.9408 0.6261 0.9513 0.6487
      0.70       0.6866       0.9317 0.5970 0.9528 0.6387

=== MLP — top 5 thresholds by F1 ===
 threshold  sensitivity  specificity    ppv    npv     f1
      0.50       0.7704       0.8964 0.5228 0.9636 0.6229
      0.60       0.7224       0.9120 0.5474 0.9571 0.6228
      0.55       0.7449       0.9039 0.5332 0.9601 0.6215
      0.65       0.6918       0.9208 0.5628 0.9530 0.6206
      0.45       0.7939       0.8871 0.5091 0.9669 0.6203


### 4.1 Clinical Operating Points

Two thresholds are particularly relevant clinically:

- **High-sensitivity point (≥ 85% sensitivity):** for screening, where missing
  an arrhythmia is much worse than a false alarm.
- **Balanced point (max F1):** for triage, where both false positives and
  false negatives carry costs.

In [13]:
def pick_thresholds(sweep_df):
    # High-sensitivity point: lowest threshold that achieves sens >= 0.85
    hs = sweep_df[sweep_df["sensitivity"] >= 0.85]
    hs_threshold = hs["threshold"].max() if len(hs) else None
    hs_row = hs.loc[hs["threshold"].idxmax()] if len(hs) else None

    # Balanced point: max F1
    bal_row = sweep_df.loc[sweep_df["f1"].idxmax()]

    return {
        "high_sens": {"threshold": hs_threshold, "row": hs_row},
        "balanced":  {"threshold": bal_row["threshold"], "row": bal_row},
    }

NUMERIC_COLS = ["threshold", "sensitivity", "specificity", "ppv", "npv", "f1",
                "tp", "fp", "tn", "fn"]

for name, sweep in [("RandomForest", rf_sweep), ("MLP", mlp_sweep)]:
    picks = pick_thresholds(sweep)
    print(f"\n=== {name} ===")
    print(f"High-sensitivity point (sens >= 0.85): threshold = {picks['high_sens']['threshold']}")
    if picks["high_sens"]["row"] is not None:
        print(picks["high_sens"]["row"][NUMERIC_COLS].round(4).to_string())
    print(f"\nBalanced point (max F1): threshold = {picks['balanced']['threshold']}")
    print(picks["balanced"]["row"][NUMERIC_COLS].round(4).to_string())


=== RandomForest ===
High-sensitivity point (sens >= 0.85): threshold = 0.1
threshold         0.1
sensitivity    0.8621
specificity    0.7824
ppv            0.3687
npv            0.9747
f1             0.5165
tp               1832
fp               3137
tn              11279
fn                293

Balanced point (max F1): threshold = 0.85
threshold        0.85
sensitivity    0.6348
specificity    0.9618
ppv              0.71
npv             0.947
f1             0.6703
tp               1349
fp                551
tn              13865
fn                776

=== MLP ===
High-sensitivity point (sens >= 0.85): threshold = 0.2
threshold         0.2
sensitivity    0.8758
specificity    0.8431
ppv            0.4514
npv            0.9787
f1             0.5957
tp               1861
fp               2262
tn              12154
fn                264

Balanced point (max F1): threshold = 0.49999999999999994
threshold         0.5
sensitivity    0.7704
specificity    0.8964
ppv            0.5228
npv   

## 5. Final Test Evaluation

The test split (7 held-out records, 15,301 beats, 7.85% abnormal) is evaluated
**once** using the two clinical operating points selected on validation:

- **Screening point (high sensitivity):** MLP threshold=0.20, RF threshold=0.10.
- **Triage point (max F1):** MLP threshold=0.50, RF threshold=0.85.

The test set was never used for model selection or threshold tuning. This
guarantees an honest estimate of generalization.

In [14]:
# Get test probabilities for the models
lr_proba_test = lr_model.predict_proba(X_test_s)[:, 1]
rf_proba_test = rf_model.predict_proba(X_test_s)[:, 1]
gb_proba_test = gb_model.predict_proba(X_test_s)[:, 1]

# Use the best seed's MLP
best_model = best_run["model"]
mlp_proba_test = predict_mlp_proba(best_model, X_test_s)

# Define clinical operating points (from validation)
OPERATING_POINTS = {
    "LogReg":            {"screening": 0.5,  "triage": 0.5},
    "RandomForest":      {"screening": 0.10, "triage": 0.85},
    "GradientBoosting":  {"screening": 0.5,  "triage": 0.5},
    "MLP":               {"screening": 0.20, "triage": 0.50},
}

test_probas = {
    "LogReg":           lr_proba_test,
    "RandomForest":     rf_proba_test,
    "GradientBoosting": gb_proba_test,
    "MLP":              mlp_proba_test,
}

# Evaluate each model at both operating points
rows = []
for model_name, proba in test_probas.items():
    for op_name, threshold in OPERATING_POINTS[model_name].items():
        m = evaluate_binary(y_test, proba, threshold=threshold)
        rows.append({
            "model":       model_name,
            "operating":   op_name,
            "threshold":   threshold,
            "roc_auc":     m["roc_auc"],
            "pr_auc":      m["pr_auc"],
            "sensitivity": m["sensitivity"],
            "specificity": m["specificity"],
            "ppv":         m["ppv"],
            "npv":         m["npv"],
            "f1":          m["f1"],
            "tp": m["tp"], "fp": m["fp"], "tn": m["tn"], "fn": m["fn"],
        })

test_results = pd.DataFrame(rows)
print("=== TEST SET RESULTS (held-out, evaluated once) ===")
print(test_results[["model", "operating", "threshold", "roc_auc", "pr_auc",
                     "sensitivity", "specificity", "ppv", "npv", "f1"]].round(4).to_string(index=False))

=== TEST SET RESULTS (held-out, evaluated once) ===
           model operating  threshold  roc_auc  pr_auc  sensitivity  specificity    ppv    npv     f1
          LogReg screening       0.50   0.8658  0.4402       0.7344       0.8960 0.3756 0.9754 0.4970
          LogReg    triage       0.50   0.8658  0.4402       0.7344       0.8960 0.3756 0.9754 0.4970
    RandomForest screening       0.10   0.9431  0.6357       0.9234       0.8170 0.3006 0.9921 0.4536
    RandomForest    triage       0.85   0.9431  0.6357       0.2565       0.9926 0.7458 0.9400 0.3817
GradientBoosting screening       0.50   0.9126  0.3481       0.6461       0.9193 0.4054 0.9683 0.4982
GradientBoosting    triage       0.50   0.9126  0.3481       0.6461       0.9193 0.4054 0.9683 0.4982
             MLP screening       0.20   0.9166  0.5962       0.8293       0.9028 0.4210 0.9842 0.5585
             MLP    triage       0.50   0.9166  0.5962       0.7194       0.9394 0.5029 0.9752 0.5920


### 5.1 Clinical Decision: Which Model to Deploy?

The deployment decision depends on the clinical use case. For **ambulatory ECG
screening**, the priority is to minimize missed arrhythmias (maximize sensitivity)
even at the cost of false alarms. For **triage in a cardiology clinic**, a
balanced threshold is preferable to reduce alarm fatigue.

The table below highlights the best model at each operating point.

In [15]:
# Best per operating point (by F1)
for op in ["screening", "triage"]:
    subset = test_results[test_results["operating"] == op]
    best = subset.loc[subset["f1"].idxmax()]
    print(f"\n=== Best model at {op.upper()} point ===")
    print(f"Model: {best['model']}")
    print(f"Threshold: {best['threshold']:.2f}")
    print(f"Sensitivity: {best['sensitivity']:.4f}")
    print(f"Specificity: {best['specificity']:.4f}")
    print(f"PPV: {best['ppv']:.4f}")
    print(f"NPV: {best['npv']:.4f}")
    print(f"F1: {best['f1']:.4f}")
    print(f"Confusion: TP={best['tp']}, FP={best['fp']}, TN={best['tn']}, FN={best['fn']}")


=== Best model at SCREENING point ===
Model: MLP
Threshold: 0.20
Sensitivity: 0.8293
Specificity: 0.9028
PPV: 0.4210
NPV: 0.9842
F1: 0.5585
Confusion: TP=996, FP=1370, TN=12730, FN=205

=== Best model at TRIAGE point ===
Model: MLP
Threshold: 0.50
Sensitivity: 0.7194
Specificity: 0.9394
PPV: 0.5029
NPV: 0.9752
F1: 0.5920
Confusion: TP=864, FP=854, TN=13246, FN=337


### 5.2 Test vs Validation Generalization

The test split has a lower abnormality rate (7.85%) than validation (12.85%),
making it a harder evaluation. A small drop in performance is expected and
indicates honest generalization rather than overfitting.

In [16]:
# Compare best MLP val vs test metrics
val_metrics = best_run["metrics"]
test_mlp_screen = test_results[(test_results["model"] == "MLP") &
                                (test_results["operating"] == "screening")].iloc[0]

comparison = pd.DataFrame([
    {"metric": "ROC-AUC",     "validation": val_metrics["roc_auc"],     "test": test_mlp_screen["roc_auc"]},
    {"metric": "PR-AUC",      "validation": val_metrics["pr_auc"],      "test": test_mlp_screen["pr_auc"]},
    {"metric": "Sensitivity", "validation": val_metrics["sensitivity"], "test": test_mlp_screen["sensitivity"]},
    {"metric": "Specificity", "validation": val_metrics["specificity"], "test": test_mlp_screen["specificity"]},
    {"metric": "F1",          "validation": val_metrics["f1"],          "test": test_mlp_screen["f1"]},
])
comparison["delta"] = comparison["test"] - comparison["validation"]
print(comparison.round(4).to_string(index=False))

     metric  validation   test   delta
    ROC-AUC      0.9268 0.9166 -0.0102
     PR-AUC      0.6248 0.5962 -0.0286
Sensitivity      0.7704 0.8293  0.0590
Specificity      0.8964 0.9028  0.0065
         F1      0.6229 0.5585 -0.0645


## 6. Persist Results and Checkpoints

All metrics, thresholds, and model checkpoints are saved to `results/` for
reproducibility and for use in the final report (FASE 4).

In [17]:
import json
from pathlib import Path

RESULTS_DIR = Path("../results")
CKPT_DIR = RESULTS_DIR / "checkpoints"
FIG_DIR = RESULTS_DIR / "figures"
for d in [RESULTS_DIR, CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Save test metrics table
test_results.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
print(f" Saved: {RESULTS_DIR / 'test_metrics.csv'}")

# Save MLP checkpoint (best seed)
torch.save({
    "state_dict": best_run["model"].state_dict(),
    "config":   config,
    "seed":     best_run["seed"],
    "feature_columns": FEATURE_COLUMNS,
    "thresholds": {"screening": 0.20, "triage": 0.50},
    "val_metrics": best_run["metrics"],
}, CKPT_DIR / "mlp_best.pt")
print(f" Saved: {CKPT_DIR / 'mlp_best.pt'}")

# Save baseline models with pickle (sklearn)
import pickle
with open(CKPT_DIR / "baselines.pkl", "wb") as f:
    pickle.dump({
        "logreg": lr_model,
        "random_forest": rf_model,
        "gradient_boosting": gb_model,
        "scalers": scalers,
    }, f)
print(f" Saved: {CKPT_DIR / 'baselines.pkl'}")

# Save summary JSON
summary = {
    "best_model": "MLP",
    "best_seed": int(best_run["seed"]),
    "operating_points": {"screening": 0.20, "triage": 0.50},
    "test_metrics": {
        "MLP_screening": test_results[(test_results["model"]=="MLP") & (test_results["operating"]=="screening")].iloc[0].to_dict(),
        "MLP_triage":    test_results[(test_results["model"]=="MLP") & (test_results["operating"]=="triage")].iloc[0].to_dict(),
    },
    "baseline_models": ["LogReg", "RandomForest", "GradientBoosting"],
    "dataset": {"n_train": len(y_train), "n_val": len(y_val), "n_test": len(y_test)},
}
# Convert numpy types to Python native
for k in ["MLP_screening", "MLP_triage"]:
    for kk, vv in summary["test_metrics"][k].items():
        if hasattr(vv, "item"):
            summary["test_metrics"][k][kk] = vv.item()

with open(RESULTS_DIR / "test_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f" Saved: {RESULTS_DIR / 'test_summary.json'}")

 Saved: ..\results\test_metrics.csv
 Saved: ..\results\checkpoints\mlp_best.pt
 Saved: ..\results\checkpoints\baselines.pkl
 Saved: ..\results\test_summary.json


## 7. Conclusions of FASE 2

### Key results

1. **The MLP transfer is successful.** Despite starting from a housing
   regression model, the Workshop 2 architecture + conditioning protocol
   adapted to binary ECG classification achieves the best F1 at both
   screening (0.559) and triage (0.592) operating points.

2. **Random Forest has a calibration crisis.** Its ROC-AUC (0.943) is the
   highest, but it cannot be deployed at any single threshold without
   sacrificing either sensitivity (triage: 26%) or PPV (screening: 30%).
   The MLP balances both.

3. **Threshold selection is clinically decisive.** The same model can be
   "useless" or "deployable" depending on the operating point. This
   reinforces the Workshop 6 requirement that threshold analysis be treated
   as first-class, not as an afterthought.

4. **Generalization is honest.** Test ROC-AUC (0.917) is within 0.01 of
   validation (0.927) despite the test split having a lower abnormality
   rate (7.85% vs 12.85%). No overfitting was detected.

### What comes in FASE 3

- **Workshop 4 transfer:** autoencoder trained on Normal beats, used as an
  unsupervised anomaly score. Compare its ROC-AUC against the supervised MLP.
- **Workshop 5 transfer:** WGAN-GP to generate synthetic abnormal beats,
  addressing the critical rarity of Fusion beats (0.79% of the dataset).
- **Ablation study:** isolate the contribution of class weighting,
  architecture depth, and preprocessing.
- **Interpretability:** Grad-CAM on ECG spectrograms, feature attribution
  on the tabular MLP.
- **Final report:** assemble the 12-section document per Workshop 6 spec.